In [1]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk, simpledialog
import os
import pandas as pd
import plotly.graph_objects as go
import openpyxl
import win32com.client as win32
import time
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
import math
import sys
import traceback
import webbrowser

class ExcelGraphPlotter:
    def __init__(self, root):
        self.root = root
        self.root.title("Brake curve plotter")
        self.saved_excel_path = None
        self.location_values = {}
        # Update these paths to match your machine
        self.firefox_path = r"C:\Users\Raghavendra\AppData\Local\Mozilla Firefox\firefox.exe" # update as needed
        self.geckodriver_path = r"C:\Tools\geckodriver\geckodriver.exe"     # update as needed
    
        # Use ttk style
        style = ttk.Style()
        try:
            style.theme_use('vista')
        except:
            pass
    
        # --- SCROLLABLE FRAME SETUP ---
        outer_frame = ttk.Frame(self.root)
        outer_frame.pack(fill=tk.BOTH, expand=True)
    
        self.canvas = tk.Canvas(outer_frame, borderwidth=0, highlightthickness=0)
        self.canvas.pack(side="left", fill="both", expand=True, padx=(0, 2))
    
        self.scrollbar = ttk.Scrollbar(outer_frame, orient="vertical", command=self.canvas.yview)
        self.scrollbar.pack(side="right", fill="y")
    
        self.canvas.configure(yscrollcommand=self.scrollbar.set)
    
        self.scrollable_frame = ttk.Frame(self.canvas)
        self.scrollable_window = self.canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
    
        def update_scrollregion(event):
            self.canvas.configure(scrollregion=self.canvas.bbox("all"))
            self.canvas.itemconfig(self.scrollable_window, width=self.canvas.winfo_width())
    
        self.scrollable_frame.bind("<Configure>", update_scrollregion)
    
        # Mouse wheel scroll binding
        self.scrollable_frame.bind("<Enter>", lambda _: self._bind_mousewheel())
        self.scrollable_frame.bind("<Leave>", lambda _: self._unbind_mousewheel())
    
        # --- GUI WIDGETS INSIDE SCROLLABLE FRAME ---
        self.labl_excelinfo = tk.Label(self.scrollable_frame, text="Load the excel file")
        self.labl_excelinfo.pack(pady=2)
    
        btn_browse = ttk.Button(self.scrollable_frame, text="Browse Excel File", command=self.browse_file)
        btn_browse.pack(pady=1, padx=75)
    
        self.location_frame = ttk.Frame(self.scrollable_frame)
        self.location_frame.pack(pady=15)
    
        self.btn_ok = ttk.Button(self.scrollable_frame, text="OK", command=self.confirm_locations)
        self.btn_ok.pack(pady=15)
        self.btn_ok.config(state=tk.DISABLED)
            
        self.btn_draw_plotly = ttk.Button(self.scrollable_frame, text="Add HTML to Excel (Generate HTML)", command=self.draw_graphs)
        self.btn_draw_plotly.pack(pady=8)
        self.btn_draw_plotly.config(state=tk.DISABLED)

        # Open graphs folder directly
        self.btn_open_graphs = ttk.Button(self.scrollable_frame, text="Open graphs folder", command=self.open_graphs_folder)
        self.btn_open_graphs.pack(pady=4)
        self.btn_open_graphs.config(state=tk.DISABLED)

        # Loading indicator for Plotly
        self.plotly_loader = ttk.Label(self.scrollable_frame, text="⏳ Generating HTML files...", foreground="blue")
        self.plotly_loader.pack()
        self.plotly_loader.pack_forget()
    
        self.btn_draw_matplotlib = ttk.Button(self.scrollable_frame, text="Add Images to Excel (Take screenshots & insert)", command=self.draw_graphs_matplotlib)
        self.btn_draw_matplotlib.pack(pady=8)
        self.btn_draw_matplotlib.config(state=tk.DISABLED)

        # Loading indicator for Images
        self.image_loader = ttk.Label(self.scrollable_frame, text="⏳ Taking screenshots and inserting into Excel...", foreground="blue")
        self.image_loader.pack()
        self.image_loader.pack_forget()
    
        self.btn_save = ttk.Button(self.scrollable_frame, text="Save & Close", command=self.closeapp)
        self.btn_save.pack(pady=15)
        self.btn_save.config(state=tk.DISABLED)

    def _bind_mousewheel(self):
        self.canvas.bind_all("<MouseWheel>", self._on_mousewheel)
    
    def _unbind_mousewheel(self):
        self.canvas.unbind_all("<MouseWheel>")
    
    def _on_mousewheel(self, event):
        self.canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")

    def browse_file(self):
        filename = filedialog.askopenfilename(filetypes=[("Excel files", "*.xlsx *.xls")])
        if filename:
            self.saved_excel_path = filename
            self.ask_for_locations()

    def closeapp(self):
        messagebox.showinfo("Success", "Process complete. Closing application.")
        self.root.destroy()

    def open_graphs_folder(self):
        if hasattr(self, 'graphs_folder') and os.path.exists(self.graphs_folder):
            try:
                # Windows
                if sys.platform.startswith('win'):
                    os.startfile(self.graphs_folder)
                else:
                    webbrowser.open(self.graphs_folder)
            except Exception as e:
                messagebox.showerror("Error", f"Could not open folder: {str(e)}")
        else:
            messagebox.showwarning("Warning", "Graphs folder not found. Generate HTML first.")

    def ask_for_locations(self):
        # Clear previous location entries
        for widget in self.location_frame.winfo_children():
            widget.destroy()
    
        # Load the workbook
        try:
            workbook = openpyxl.load_workbook(self.saved_excel_path, data_only=True)
        except Exception as e:
            messagebox.showerror("Error", f"Could not open workbook: {e}")
            return
    
        # Initialize dictionary to store location values for each sheet
        location_data = {}
    
        # Check if 'Summary' sheet exists and try to prefill values
        if 'Summary' in workbook.sheetnames:
            summary_sheet = workbook['Summary']
    
            # --- Table 1: Fixed Headers (Row 2) ---
            try:
                headers_table_1 = {cell.value: cell.column for cell in summary_sheet[2]}
                station_column_table_1 = headers_table_1.get("Station")
                signal_column_table_1 = headers_table_1.get("Signal Location")
            except Exception:
                station_column_table_1 = None
                signal_column_table_1 = None
    
            if station_column_table_1 and signal_column_table_1:
                for row in summary_sheet.iter_rows(min_row=3, max_row=summary_sheet.max_row):
                    station_value = row[station_column_table_1 - 1].value
                    if station_value in workbook.sheetnames:
                        signal_value = row[signal_column_table_1 - 1].value
                        location_data[station_value] = {"Signal Location": signal_value}
    
            # --- Table 2: Varying Headers ---
            abs_location_column_table_2 = None
            station_column_table_2 = None
    
            for row in summary_sheet.iter_rows(min_row=4, max_row=summary_sheet.max_row):
                for cell in row:
                    if cell.value == "Station":
                        station_column_table_2 = cell.column
                    if cell.value == "ABS Location of facing Turnout":
                        abs_location_column_table_2 = cell.column
                if station_column_table_2 and abs_location_column_table_2:
                    break
    
            if station_column_table_2 and abs_location_column_table_2:
                # compute proper min_row: station_column_table_2 is a column index not row - handle gracefully
                try:
                    for row in summary_sheet.iter_rows(min_row=station_column_table_2 + 1, max_row=summary_sheet.max_row):
                        station_value = row[station_column_table_2 - 1].value
                        if station_value in workbook.sheetnames:
                            abs_location_value = row[abs_location_column_table_2 - 1].value
                            if station_value in location_data:
                                location_data[station_value]["ABS Location of facing Turnout"] = abs_location_value
                            else:
                                location_data[station_value] = {"ABS Location of facing Turnout": abs_location_value}
                except Exception:
                    # fallback: iterate rows normally and try to find by header values
                    pass
        
        # Ask for location values based on sheet names and prefill if available
        self.location_entries = {}
        for sheet_name in workbook.sheetnames:
            if sheet_name == 'Summary':
                continue
    
            if sheet_name.endswith('_LL'):
                location_label = f"Loop entry location for {sheet_name}"
                prefill_value = location_data.get(sheet_name, {}).get("ABS Location of facing Turnout", "")
            else:
                location_label = f"Signal location for {sheet_name}"
                prefill_value = location_data.get(sheet_name, {}).get("Signal Location", "")
    
            label = ttk.Label(self.location_frame, text=location_label)
            label.pack(anchor='w')
            entry = ttk.Entry(self.location_frame)
            entry.pack(anchor='w')
            
            if prefill_value is not None and prefill_value != "":
                entry.insert(0, str(prefill_value))
    
            self.location_entries[sheet_name] = entry
    
        # Enable OK button
        self.btn_ok.config(state=tk.NORMAL)
    
    def confirm_locations(self):
        for sheet_name, entry in self.location_entries.items():
            location_value = entry.get()
            if location_value:
                try:
                    self.location_values[sheet_name] = float(location_value)
                except:
                    self.location_values[sheet_name] = None
            else:
                self.location_values[sheet_name] = None

        # Enable Draw Graphs buttons
        self.btn_draw_plotly.config(state=tk.NORMAL)
        self.btn_draw_matplotlib.config(state=tk.NORMAL)
        self.btn_save.config(state=tk.NORMAL)
        self.btn_open_graphs.config(state=tk.NORMAL)

#-------------------------------------------------------------------------------------------------------------
    # Draw Graphs
    def draw_graphs(self):
        if not self.saved_excel_path:
            messagebox.showerror("Error", "Please save an Excel workbook first.")
            return
    
        # Load the workbook
        try:
            workbook = openpyxl.load_workbook(self.saved_excel_path, data_only=True)
        except Exception as e:
            messagebox.showerror("Error", f"Could not open workbook: {e}")
            return
    
        # Create 'graphs' folder next to the Excel file
        excel_folder = os.path.dirname(self.saved_excel_path)
        graphs_folder = os.path.join(excel_folder, 'graphs')
        os.makedirs(graphs_folder, exist_ok=True)
        self.graphs_folder = graphs_folder
        
        try:
            self.plotly_loader.pack()
            self.root.update() 
    
            for sheet_name in workbook.sheetnames:
                if sheet_name == 'Summary':
                    continue 
                    
                sheet = workbook[sheet_name]
        
                # Convert sheet to DataFrame
                df = pd.DataFrame(sheet.values)
                if df.shape[0] < 2:
                    # no data
                    continue
                df.columns = df.iloc[0]
                df = df[1:].reset_index(drop=True)
        
                # Ensure 'Speed' and 'Distance moved' columns exist
                if 'Speed' not in df.columns or 'Distance moved' not in df.columns:
                    messagebox.showwarning("Warning", f"Sheet '{sheet_name}' does not contain 'Speed' and 'Distance moved' columns.")
                    continue
        
                # Convert columns to numeric if needed
                df['Speed'] = pd.to_numeric(df['Speed'], errors='coerce')
                df['Distance moved'] = pd.to_numeric(df['Distance moved'], errors='coerce')

                # Handle missing speed values
                max_speed = df['Speed'].max()
                if pd.isna(max_speed) or not math.isfinite(max_speed):
                    max_speed = 100.0

                # Determine x-axis range based on sheet name
                if sheet_name.endswith('_LL'):
                    try:
                        last_dist = float(df['Distance moved'].iloc[-1])
                    except Exception:
                        last_dist = 0.0
                    x_range_max = last_dist + 120
                else:
                    # Compute difference in AbsLoc values
                    if 'AbsLoc' not in df.columns:
                        messagebox.showwarning("Warning", f"Sheet '{sheet_name}' does not contain 'AbsLoc' column.")
                        continue

                    abs_loc_diff = pd.to_numeric(df['AbsLoc'], errors='coerce').diff().dropna()
                    if abs_loc_diff.empty:
                        messagebox.showwarning("Warning", f"Sheet '{sheet_name}' AbsLoc diff empty.")
                        continue

                    pos_count = (abs_loc_diff > 0).sum()
                    neg_count = (abs_loc_diff < 0).sum()
                    total_count = len(abs_loc_diff)
            
                    threshold = 0.5
                    abs_loc_increasing = (pos_count / total_count) > threshold
                    abs_loc_decreasing = (neg_count / total_count) > threshold
            
                    # Get user-entered signal location
                    signal_location = self.location_values.get(sheet_name, None)
                    if signal_location is None:
                        # skip if no user-provided signal location
                        continue
            
                    try:
                        loco_halted_location = float(pd.to_numeric(df['AbsLoc'], errors='coerce').iloc[-1])
                    except Exception:
                        loco_halted_location = float(pd.to_numeric(df['AbsLoc'], errors='coerce').dropna().iloc[-1])
                    halted_distance = signal_location - loco_halted_location
            
                    # Determine annotation position & X-axis range based on conditions
                    if abs_loc_increasing:
                        if halted_distance >= 0:
                            signal_annotation_x = float(df['Distance moved'].iloc[-1]) + abs(halted_distance)
                            x_range_max = signal_annotation_x + 120
                            spad = "Success"
                        else:
                            signal_annotation_x = float(df['Distance moved'].iloc[-1]) - abs(halted_distance)
                            x_range_max = float(df['Distance moved'].iloc[-1]) + 120
                            spad = "Fail"
                    elif abs_loc_decreasing:
                        if halted_distance <= 0:
                            signal_annotation_x = float(df['Distance moved'].iloc[-1]) + abs(halted_distance)
                            x_range_max = signal_annotation_x + 120
                            spad = "Success"
                        else:
                            signal_annotation_x = float(df['Distance moved'].iloc[-1]) - abs(halted_distance)
                            x_range_max = float(df['Distance moved'].iloc[-1]) + 120
                            spad = "Fail"
                    else:
                        messagebox.showwarning("Warning", f"Sheet '{sheet_name}' does not have a clear AbsLoc direction.")
                        continue
                        
                # Create Plotly figure
                fig = go.Figure()
        
                # Add the scatter plot of data as lines+markers to the figure
                fig.add_trace(go.Scatter(x=df['Distance moved'], y=df['Speed'], mode='lines+markers', name='All Data'))
        
                # Add vertical dashed green lines based on location values
                if sheet_name in self.location_values and self.location_values[sheet_name] is not None:
                    entered_loc_value = self.location_values[sheet_name]
                    if not sheet_name.endswith('_LL'):
                        x_value = signal_annotation_x
                    else:
                        try:
                            x_value = abs(float(df['Distance moved'].iloc[-1]) - abs(float(df['AbsLoc'].iloc[-1]) - entered_loc_value))
                        except Exception:
                            x_value = 0.0
        
                    fig.add_shape(type="line",
                                  x0=x_value, y0=0, x1=x_value, y1=max_speed,
                                  line=dict(color="darkred", width=2, dash="dash"),
                                  xref='x', yref='y')
                    
                    # Add annotation for location type
                    if sheet_name.endswith('_LL'):
                        location_type = f'<b>LOOP LOCATION: {int(entered_loc_value)}</b>'
                    else:
                        location_type = f'<b>SIGNAL LOCATION: {int(entered_loc_value)}</b>'
        
                    fig.add_annotation(
                        x=x_value, y=max_speed / 1.5,
                        xref='x', yref='y',
                        text=f"{location_type}",
                        showarrow=False,
                        arrowhead=2,
                        ax=0,
                        ay=-70,
                        bgcolor='darkred',
                        bordercolor='darkred',
                        borderwidth=2,
                        borderpad = 4,
                        font=dict(color='white', size=18, family = 'Arial'),
                        align='center'
                    )

                    if not sheet_name.endswith("_LL"):
                        if spad == "Success":
                            state = "BEFORE"
                        else:
                            state = "AFTER"
                        
                        fig.add_annotation(
                            x=x_value - 150, y=max_speed - 25,
                            xref='x', yref='y',
                            text=f"<b>LOCO HALTED {state} {int(abs(halted_distance))} m</b>",
                            showarrow=False,
                            arrowhead=2,
                            ax=0,
                            ay=-95,
                            bgcolor='blue',
                            bordercolor='blue',
                            borderwidth=2,
                            borderpad = 4,
                            font=dict(color='white', size=18, family = 'Arial'),
                            align='center'
                        )
        
                # Add brake annotations
                used_annotations = []
                if 'LeBL' in df.columns or 'SBp' in df.columns:
                    if 'LeBL' in df.columns:
                        brake_conditions = {
                            4: 'FSB',
                            5: 'EB',
                            0: 'BRAKE RELEASED'
                        }
                        previous_value = None
                        for i, row in df.iterrows():
                            try:
                                lebl_val = float(row['LeBL'])
                            except Exception:
                                lebl_val = row['LeBL']
                            if lebl_val != previous_value:
                                if lebl_val in brake_conditions:
                                    ay_value = -40
                                    for annot in used_annotations:
                                        try:
                                            if abs(float(row['Distance moved']) - annot['distance']) < 50:
                                                ay_value -= 30  # Adjust height to avoid overlap
                                        except:
                                            pass
                
                                    annotation_text = f"<b>{brake_conditions[lebl_val]}: SPEED = {row['Speed']}</b>" if lebl_val != 0 else "<b>BRAKE RELEASED</b>"
                
                                    # Set color based on brake status
                                    if lebl_val == 0:
                                        bgcolor = 'green'
                                        bordercolor = 'green'
                                    else:
                                        bgcolor = 'red'
                                        bordercolor = 'red'
                
                                    fig.add_annotation(
                                        x=row['Distance moved'], y=row['Speed'],
                                        xref='x', yref='y',
                                        text=annotation_text,
                                        showarrow=True,
                                        arrowhead=2,
                                        ax=10,
                                        ay=ay_value,
                                        bgcolor=bgcolor,
                                        bordercolor=bordercolor,
                                        borderwidth=2,
                                        font=dict(color='white', size=18, family = 'Arial'),
                                        align='center'
                                    )
                                    try:
                                        used_annotations.append({'distance': float(row['Distance moved']), 'ay': ay_value})
                                    except:
                                        pass
                                    previous_value = lebl_val

                    # SBp
                    if 'SBp' in df.columns:
                        brake_conditions = {
                            3.5: 'FSB',
                            3.7: 'FSB',
                            4.2: 'NB',
                            0: 'EB',
                            5.5: 'BRAKE RELEASED'
                        }

                        previous_value = None
                        cluster_group = []

                        for i, row in df.iterrows():
                            try:
                                sbp_val = float(row['SBp'])
                            except Exception:
                                sbp_val = row['SBp']

                            if sbp_val != previous_value and sbp_val in brake_conditions:

                                distance = row['Distance moved']

                                # -----------------------------------------
                                # FIND CLOSE CLUSTER (within 50 distance)
                                # -----------------------------------------
                                close_items = [c for c in cluster_group if abs(float(distance) - c) < 50] if cluster_group else []

                                # Add to cluster memory
                                try:
                                    cluster_group.append(float(distance))
                                except:
                                    cluster_group.append(0.0)

                                cluster_index = len(close_items)

                                # -----------------------------------------
                                # DECIDE TOP/BOTTOM OFFSET BASED ON CLUSTER
                                # -----------------------------------------
                                if cluster_index % 2 == 0:
                                    ay_value = 40 + (cluster_index * 10)
                                else:
                                    ay_value = -40 - (cluster_index * 10)

                                # -----------------------------------------
                                # DECIDE SLANT BASED ON TYPE
                                # -----------------------------------------
                                if sbp_val == 5.5:
                                    ax_value = 25 + (cluster_index * 5)
                                else:
                                    ax_value = -25 - (cluster_index * 5)

                                # ----- TEXT -----
                                if sbp_val == 5.5:
                                    annotation_text = "<b>BRAKE RELEASED</b>"
                                else:
                                    annotation_text = f"<b>{brake_conditions[sbp_val]}: SPEED = {row['Speed']}</b>"

                                # ----- COLORS -----
                                if sbp_val == 5.5:
                                    bgcolor = 'green'
                                    bordercolor = 'green'
                                else:
                                    bgcolor = 'red'
                                    bordercolor = 'red'

                                # ----- ADD ANNOTATION -----
                                fig.add_annotation(
                                    x=distance, y=row['Speed'],
                                    text=annotation_text,
                                    xref='x', yref='y',
                                    showarrow=True,
                                    arrowhead=2,
                                    ax=ax_value,
                                    ay=ay_value,
                                    bgcolor=bgcolor,
                                    bordercolor=bordercolor,
                                    borderwidth=2,
                                    borderpad=4,
                                    font=dict(color='white', size=18, family='Arial')
                                )

                                previous_value = sbp_val

                    # Braking distance line & annotation for non-loop sheets
                    if not sheet_name.endswith('_LL'):
                        try:
                            x1_range = float(df["Distance moved"].iloc[-1])
                        except:
                            x1_range = 0.0
                        fig.add_shape(type="line", x0=0, y0=20, x1=x1_range, y1=20,
                                      line=dict(color="rgb(0, 111, 0)", width=1.5), xref='x', yref='y')              
                        fig.add_annotation(x=x1_range/2, y=20, xref='x', yref='y',
                                           text=f"<b>BRAKING DISTANCE = {int(x1_range)} m</b>", showarrow=False,
                                           bgcolor="rgb(0, 111, 0)", bordercolor="rgb(0, 111, 0)", borderwidth=2, borderpad=4,
                                           font=dict(color='white', size=18, family='Arial'), align='center')
                            
                    # Add horizontal dashed dark yellow line at 30 km/h for sheets ending with '_LL'
                    if sheet_name.endswith('_LL'):
                        y30 = 30
                        fig.add_shape(type="line",
                                      x0=0, y0=y30, x1=x_range_max, y1=y30,
                                      line=dict(color="rgb(255,186,0)", width=2),
                                      xref='x', yref='y')
                    
                        fig.add_annotation(
                            x=x_range_max / 2, y=y30,
                            xref='x', yref='y',
                            text='<b>TO Speed control: 30kmph</b>',
                            showarrow=False,
                            bgcolor='rgb(255,186,0)',
                            bordercolor='rgb(255,186,0)',
                            borderwidth=2,
                            borderpad = 4,
                            font=dict(color='black', size=18, family = 'Arial'),
                            align='center'
                        )
                    
                        went_second_loop = messagebox.askyesno(
                            "Loop Line Confirmation",
                            f"Did the train in sheet '{sheet_name}' go through the second loop line?"
                        )
                        
                        if went_second_loop:
                            second_loop_entry = simpledialog.askfloat(
                                "Second Loop Entry Location",
                                f"Enter the AbsLoc of second loop line in '{sheet_name}':"
                            )
                        
                            if second_loop_entry is not None:
                                y15 = 15
                                fig.add_shape(
                                    type="line",
                                    x0=0, y0=y15, x1=x_range_max, y1=y15,
                                    line=dict(color="rgb(255,186,0)", width=2),
                                    xref='x', yref='y'
                                )
                        
                                fig.add_annotation(
                                    x=x_range_max / 2, y=y15,
                                    xref='x', yref='y',
                                    text='<b>TO Speed control: 15kmph</b>',
                                    showarrow=False,
                                    bgcolor='rgb(255,186,0)',
                                    bordercolor='rgb(255,186,0)',
                                    borderwidth=2,
                                    font=dict(color='black', size=18, family = 'Arial'),
                                    align='center'
                                )
                                # second loop entry line
                                try:
                                    x_value = abs(float(df['Distance moved'].iloc[-1]) - abs(float(df['AbsLoc'].iloc[-1]) - second_loop_entry))
                                except:
                                    x_value = 0.0
                                
                                fig.add_shape(
                                    type="line",
                                    x0=x_value, y0=0,
                                    x1=x_value, y1=max_speed,
                                    line=dict(color="darkred", width=2, dash="dash"),
                                    xref='x', yref='y'
                                )
                                
                                fig.add_annotation(
                                    x=x_value,
                                    y=max_speed / 1.2,
                                    xref='x', yref='y',
                                    text=f"<b>SECOND LOOP ENTRY: {int(second_loop_entry)}</b>",
                                    showarrow=False,
                                    arrowhead=2,
                                    ax=0,
                                    ay=-70,
                                    bgcolor='darkred',
                                    bordercolor='darkred',
                                    borderwidth=2,
                                    font=dict(color='white', size=18, family = 'Arial'),
                                    align='center'
                                )
                
                # Configure layout (do NOT set editable here - it will be enabled via config when saving)
                fig.update_layout(
                    title=sheet_name,
                    font=dict(size=14, family='Arial'),
                    xaxis=dict(
                        title='Distance moved',
                        range=[-20, x_range_max],
                        showline=True,
                        linecolor='black',
                        linewidth=2,
                        zeroline=True,
                        zerolinecolor='black',
                        zerolinewidth=2,
                        gridcolor='lightgray'
                    ),
                    yaxis=dict(
                        title='Speed',
                        range=[-20, max_speed * 1.1],
                        showline=True,
                        linecolor='black',
                        linewidth=2,
                        zeroline=True,
                        zerolinecolor='black',
                        zerolinewidth=2,
                        gridcolor='lightgray'
                    )
                )
        
                # Save the graph as {sheet_name}.html in 'graphs' folder
                graph_html_path = os.path.join(graphs_folder, f"{sheet_name}.html")
                # IMPORTANT: enable editable behaviour in the HTML via config
                fig.write_html(
                    graph_html_path,
                    include_plotlyjs='cdn',
                    full_html=True,
                    config={
                        "editable": True,

                        # Enable Plotly download button
                        "displayModeBar": True,

                        # Download image settings
                        "toImageButtonOptions": {
                            "format": "png",      # png / svg / jpeg / webp
                            "filename": sheet_name,
                            "height": 1080,
                            "width": 1920,
                            "scale": 2            # Higher quality
                        },

                        # Optional: keep only useful buttons
                        "modeBarButtonsToRemove": [
                            "lasso2d",
                            "select2d",
                            "zoomIn2d",
                            "zoomOut2d",
                            "autoScale2d",
                            "toggleSpikelines"
                        ],

                        "edits": {
                            "titleText": False,
                            "legendText": False,
                            "annotationPosition": True,
                            "shapePosition": True
                        }
                    }
                )
       
            # Show instruction message once (do not open many popups)
            messagebox.showinfo(
                "Manual Editing Required",
                "All HTML graphs were generated in the 'graphs' folder.\n\n"
                "Please open each HTML file from the 'graphs' folder in your browser, manually adjust annotations (drag them) as needed,\n"
                "then return to this app and click 'Add Images to Excel' to take screenshots of the edited graphs and insert them into Excel.\n\n"
                "Tip: Use the 'Open graphs folder' button to open the folder quickly."
            )
    
        except Exception as e:
            traceback.print_exc()
            messagebox.showerror("Error", f"Error occurred while creating HTML graphs:\n{str(e)}")
    
        finally:
            self.plotly_loader.pack_forget()

#-----------------------------------------------------------------------------------------------------------------------------------

    def draw_graphs_matplotlib(self):
        firefox_path = self.firefox_path
        geckodriver_path = self.geckodriver_path
        if not self.saved_excel_path:
            messagebox.showerror("Error", "Please save an Excel workbook first.")
            return
    
        if not hasattr(self, 'graphs_folder'):
            messagebox.showerror("Error", "Graphs folder path not set. Run HTML generation first.")
            return
    
        graphs_folder = self.graphs_folder
    
        try:
            self.image_loader.pack()
            self.root.update_idletasks()
            
            # Create folder for screenshots
            excel_folder = os.path.dirname(self.saved_excel_path)
            images_folder = os.path.join(excel_folder, 'images')
            os.makedirs(images_folder, exist_ok=True)
    
            # Set up Firefox WebDriver (headless)
            options = Options()
            options.add_argument("--headless")
            options.add_argument("--width=1920")
            options.add_argument("--height=1080")
            options.binary_location = firefox_path

            service = Service(executable_path=geckodriver_path)
            driver = webdriver.Firefox(service=service, options=options)
    
            # Read sheet names
            wb = openpyxl.load_workbook(self.saved_excel_path, data_only=True)
            sheet_names = wb.sheetnames
            wb.close()
    
            # Open Excel via COM
            excel_app = win32.Dispatch("Excel.Application")
            excel_app.Visible = False
            workbook_com = excel_app.Workbooks.Open(self.saved_excel_path)
    
            for sheet_name in sheet_names:
                if sheet_name == "Summary":
                    continue
    
                html_file = os.path.join(graphs_folder, f"{sheet_name}.html")
                if not os.path.exists(html_file):
                    messagebox.showwarning("Warning", f"HTML not found: {html_file}")
                    continue
    
                # Open file:// URL
                driver.get("file://" + os.path.abspath(html_file))
                # give some time for JS and plot to render
                time.sleep(2)
    
                screenshot_path = os.path.join(images_folder, f"{sheet_name}.png")
                driver.save_screenshot(screenshot_path)
    
                if not os.path.exists(screenshot_path):
                    messagebox.showwarning("Warning", f"Screenshot not saved: {screenshot_path}")
                    continue
    
                try:
                    xl_sheet = workbook_com.Sheets(sheet_name)
                    xl_sheet.Activate()
                    if sheet_name.endswith('_LL'):
                        left_col = 16
                    else:
                        left_col = 14

                    xl_sheet.Shapes.AddPicture(
                        Filename=os.path.abspath(screenshot_path),
                        LinkToFile=False,
                        SaveWithDocument=True,
                        Left=xl_sheet.Cells(2, left_col).Left,
                        Top=xl_sheet.Cells(2, left_col).Top,
                        Width=800,
                        Height=400
                    )

                except Exception as e:
                    messagebox.showerror("Error", f"Could not insert image in {sheet_name}: {str(e)}")
                    continue
    
            workbook_com.Save()
            workbook_com.Close()
            excel_app.Quit()
            driver.quit()
    
            messagebox.showinfo("Success", "Image graphs (screenshots) inserted into Excel!")
    
        except Exception as e:
            traceback.print_exc()
            messagebox.showerror("Fatal Error", f"Unexpected error:\n{str(e)}")
            try:
                if 'excel_app' in locals():
                    excel_app.Quit()
            except:
                pass
            try:
                if 'driver' in locals():
                    driver.quit()
            except:
                pass

        finally:
            self.image_loader.pack_forget()

if __name__ == "__main__":
    root = tk.Tk()
    plotter = ExcelGraphPlotter(root)
    root.mainloop()
